###DAY 5 (24/02/26) – Production-Grade Feature Engineering
####🏗️ Architecture & Strategy
Welcome to Phase 2: AI System Building! Up to this point, we have engineered data. Now, we pivot to framing a Machine Learning problem. To train an ML model to predict user behavior, we need a Supervised Learning dataset consisting of **Features** (the X variables we built on Day 2) and a **Target Label** (the Y variable we will build today).

####Our Strategy:

* **Target Creation (The "Y" Variable)**: We need to create a binary purchase label. We will scan the raw events and flag a user with `1` if they ever made a purchase, and `0` if they only viewed or carted items.

* **The Feature Join**: We will join this label table with our `silver_user_features` table.

* **Reproducible Splitting**: We will split the train/test datasets using a fixed seed (42). In production MLOps, reproducibility is mandatory so that changes in model performance are due to code changes, not data shuffling.

* **Imbalance Validation**: In eCommerce, 95%+ of users do not buy. We must explicitly validate this distribution.  If we skip this, an ML model might just predict "0" for everyone and claim 95% accuracy!

####Target Label Creation
We start by extracting the "Ground Truth" from our event logs. Did the user ultimately make a purchase?

In [0]:
from pyspark.sql import functions as F

# 1. Setup Context
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"

print(f"🔄 Setting context to: {catalog_name}.{schema_name}")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# 2. Load the base events table (to figure out who purchased)
df_events = spark.table("events_delta_managed")

# 3. Create the Binary Target Label
print("🎯 Generating binary target labels (1 = Purchased, 0 = No Purchase)...")

# If a user has ANY purchase event, the max value will be 1. 
# If they only have views/carts, the max value remains 0.
df_labels = df_events.groupBy("user_id").agg(
    F.max(
        F.when(F.col("event_type") == "purchase", 1).otherwise(0)
    ).alias("purchased")
)

print(f"✅ Created labels for {df_labels.count():,} unique users.")
display(df_labels.limit(5)) 

####Joining Features & Train/Test Split
We now merge our Day 2 behavioral features with our Day 5 labels. Then, we split the data 80/20 for training and testing.

In [0]:
# ---------------------------------------------------------
# FEATURE JOIN & DATA SPLITTING
# ---------------------------------------------------------
print("🔗 Joining Silver Features with Target Labels...")

# 1. Load the Silver Feature Table (created on Day 2)
df_features = spark.table("silver_user_features")

# 2. Join Features (X) with Labels (Y)
# We use an inner join to ensure we only train on users where we have both features and a known outcome.
df_model_input = df_features.join(df_labels, on="user_id", how="inner")

# 3. Train / Test Split
print("✂️ Splitting data into Training (80%) and Testing (20%) sets...")

# CRITICAL MLOPS BEST PRACTICE: Always use a seed for randomSplit.
# If you don't, every run of this notebook will yield different test metrics!
df_train, df_test = df_model_input.randomSplit([0.8, 0.2], seed=42)

print(f"   ➤ Training Set: {df_train.count():,} rows")
print(f"   ➤ Testing Set:  {df_test.count():,} rows")

####Validate Distribution & Persist to Gold
Before we let an ML model touch this data, we must inspect the class imbalance. Then, we save these splits as Delta tables so our downstream MLflow pipelines can query them instantly.

In [0]:
# ---------------------------------------------------------
# CLASS IMBALANCE VALIDATION
# ---------------------------------------------------------
print("📊 Validating Class Distribution (Target Imbalance)...")

# Calculate the distribution of the target variable in the training set
distribution_df = df_train.groupBy("purchased").count().withColumn(
    "percentage", 
    F.round((F.col("count") / df_train.count()) * 100, 2)
)

display(distribution_df)

# 💡 UI INSTRUCTIONS FOR VISUALIZATION:
# 1. Click '+' -> 'Visualization' -> 'Pie Chart'.
# 2. Keys: 'purchased', Values: 'count'.
# You will likely see a massive imbalance (e.g., 90%+ Non-Purchasers).

# ---------------------------------------------------------
# PERSIST TO GOLD LAYER
# ---------------------------------------------------------
train_table = f"{catalog_name}.{schema_name}.gold_train_set"
test_table = f"{catalog_name}.{schema_name}.gold_test_set"

print(f"💾 Saving finalized ML datasets to Gold Layer...")
print(f"   ➤ {train_table}")
print(f"   ➤ {test_table}")

# We write these to disk so the next MLflow notebook reads from a static, optimized snapshot
df_train.write.format("delta").mode("overwrite").saveAsTable("gold_train_set")
df_test.write.format("delta").mode("overwrite").saveAsTable("gold_test_set")

# Z-Order by the target to speed up stratified sampling later if needed
spark.sql("OPTIMIZE gold_train_set ZORDER BY (purchased)")
spark.sql("OPTIMIZE gold_test_set ZORDER BY (purchased)")

print("✅ Day 5 Complete! Data is ready for Model Training.")

Databricks visualization. Run in Databricks to view.

####Key Learnings & Interview Talking Points
If a recruiter asks about your feature engineering or ML data prep workflow, mention these concepts:

* **Supervised Target Framing**: "I translated a business problem ('Who will buy?') into a supervised ML problem by aggregating event streams into a distinct binary label (1/0) using PySpark windowing/aggregation."

* **Handling Data Splitting Correctly**: "In my ML pipelines, I strictly enforce a seed during train/test splits. Without seeded reproducibility, model evaluation becomes a moving target, making true MLOps impossible."

* **Validating Class Imbalance**: "I actively profile my target distributions. In eCommerce datasets, the purchase class is heavily minority (often < 5%). By catching this early, I know I will need to use metrics like F1-Score or PR-AUC rather than raw accuracy when evaluating my models."

* **Persisting ML Datasets (Gold Layer)**: "Instead of re-running the join and split operations every time I tune a hyperparameter, I save the train and test dataframes as static Delta tables. This guarantees that my MLflow experiments are tracking models trained on the exact same data bytes.